<!-- ============================================================ -->
<!-- NOTEBOOK HEADER — MLOps Introductory Course on GCP           -->
<!-- ============================================================ -->

<div style="border-bottom: 3px solid #4285F4; padding-bottom: 12px; margin-bottom: 20px;">

<div style="display: flex; align-items: center; justify-content: space-between;">
  <div>
    <img src="https://www.isae-supaero.fr/wp-content/uploads/2025/03/logo.svg" width="180">
  </div>
  <div style="text-align: right;">
    <img src="https://user-images.githubusercontent.com/63151412/167391313-4683cc69-2bf6-4597-b767-5c18e2bbbfa0.png" width="180">
  </div>
</div>

# Lab 04a — Kubeflow Pipelines Fundamentals — ✅ Solution

**Course:** MLOps Introductory Course on GCP · M2 Data Science · ISAE-SUPAERO  
**Lab created by:** Headmind Partners AI & Blockchain  
**Estimated duration:** ~1h00

</div>

## 📋 Lab Overview

In the previous labs you trained models, tracked experiments, and deployed endpoints — all as individual steps. In production, these steps must be **orchestrated** into reproducible, automated **ML pipelines**. **Kubeflow Pipelines (KFP)** is a framework-agnostic pipeline SDK that integrates natively with **Vertex AI Pipelines**, Google Cloud's fully managed pipeline orchestrator.

### Learning Objectives

1. Understand the core concepts of Kubeflow Pipelines: **components**, **pipelines**, and **artifacts**.
2. Build **lightweight Python components** using the `@dsl.component` decorator.
3. Compose components into a **pipeline DAG** and **compile** it to YAML.
4. Submit and monitor a pipeline run on **Vertex AI Pipelines**.
5. **Parameterize** a pipeline and re-run it with different hyperparameters.
6. Compare pipeline runs in the Vertex AI console.

### Notebook Structure

| # | Section | Focus |
|---|---------|-------|
| 0 | Setup | Install KFP SDK, imports, GCP configuration |
| 1 | KFP Concepts | Components, pipelines, artifacts, and the DAG model |
| 2 | Building Components | Create data-prep, training, and evaluation components |
| 3 | Compose & Compile | Wire components into a pipeline and compile to YAML |
| 4 | Run on Vertex AI | Submit the pipeline and inspect the run in the console |
| 5 | Parameterize & Re-run | Re-run with different hyperparameters and compare |
| 6 | Cleanup | Delete pipeline runs |

### How to Read This Notebook

- **`# TODO`** — Code you need to write. Look for the `######` delimiters.
- **`✏️ Question`** — A conceptual question. Write your answer in the markdown cell below it.
- Cells **without** a TODO are provided — read them, run them, and make sure you understand them.
- Documentation links are provided in 📖 callouts whenever a new API is introduced.

---
## 0 · Setup

### 0.1 Install dependencies

In [ ]:
%pip install --upgrade --quiet kfp google-cloud-aiplatform google-cloud-pipeline-components

### 0.2 Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

from kfp import dsl
from kfp.dsl import Input, Output, Dataset, Model, Metrics
from kfp import compiler

import google.cloud.aiplatform as aiplatform

import kfp
print(f"KFP SDK version: {kfp.__version__}")
print(f"Vertex AI SDK version: {aiplatform.__version__}")

### 0.3 Configuration

Replace the placeholders below with your own GCP project details.

In [ ]:
# ── Constants ──
PROJECT_ID = "your-project-id"          # @param {type:"string"}
LOCATION = "europe-west3"               # @param {type:"string"}
BUCKET_URI = "gs://your-bucket-name"     # @param {type:"string"}

##############################  TODO  ##############################
# Set your_name to a unique lowercase identifier (e.g. first letter of first name + last name).
your_name = "dupont"  # TODO
####################################################################

# Pipeline artifacts will be stored under this root
PIPELINE_ROOT = f"{BUCKET_URI}/pipeline-root/lab04a/{your_name}"

# Initialize the Vertex AI SDK
aiplatform.init(
    project=PROJECT_ID,
    location=LOCATION,
    staging_bucket=BUCKET_URI,
)
print(f"✅ Vertex AI initialized — project={PROJECT_ID}, location={LOCATION}")
print(f"   Pipeline root: {PIPELINE_ROOT}")

> 📖 **Docs:** [`aiplatform.init()`](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform#google_cloud_aiplatform_init)

In [ ]:
PIPELINE_DISPLAY_NAME = f"lab04a-covertype-{your_name}"
print(f"Pipeline display name: {PIPELINE_DISPLAY_NAME}")

---
## 1 · Kubeflow Pipelines — Key Concepts

Before writing code, let's understand the three core abstractions of the KFP SDK:

**Component** — A self-contained unit of work (a Python function or a container image). Each component receives typed inputs and produces typed outputs. Think of it as a single step in your ML workflow: *load data*, *train model*, *evaluate*, etc.

**Pipeline** — A **directed acyclic graph (DAG)** of components. The pipeline defines the execution order by wiring outputs of one component to inputs of the next. KFP infers the DAG from these data dependencies.

**Artifact** — A typed object (dataset, model, metrics) that flows between components. KFP handles serialization and storage in Cloud Storage automatically.

The typical workflow is:

1. **Define** components with `@dsl.component`
2. **Compose** them into a pipeline with `@dsl.pipeline`
3. **Compile** the pipeline to a YAML file with `compiler.Compiler().compile()`
4. **Run** the YAML on Vertex AI Pipelines with `aiplatform.PipelineJob`

> 📖 **Docs:**
> - [KFP SDK v2 overview](https://www.kubeflow.org/docs/components/pipelines/user-guides/components/)
> - [Vertex AI Pipelines introduction](https://cloud.google.com/vertex-ai/docs/pipelines/introduction)

---
## 2 · Building Pipeline Components

We will build three components for a simple ML pipeline using the **Covertype** dataset (multi-class classification of forest cover types from cartographic features):

1. **`load_and_split_data`** — Fetch the dataset, subsample it, and split into train/test sets.
2. **`train_model`** — Train a scikit-learn classifier on the training set.
3. **`evaluate_model`** — Evaluate the model on the test set and log metrics.

Each component is a regular Python function decorated with `@dsl.component`. The decorator specifies the base Docker image and any pip packages the function needs.

> 📖 **Docs:** [`@dsl.component` decorator](https://www.kubeflow.org/docs/components/pipelines/user-guides/components/lightweight-python-components/)

> 💡 **Tip:** Inside a `@dsl.component` function, all imports must be **inside the function body** — the function is serialized and executed in a separate container that knows nothing about your notebook environment.

### 2.1 Data preparation component

This component is **provided**. Read it carefully — it demonstrates:
- How to declare typed outputs (`Output[Dataset]`)
- How imports go inside the function
- How to write data to artifact paths

In [ ]:
@dsl.component(
    base_image="python:3.10",
    packages_to_install=["scikit-learn==1.5.2", "pandas==2.2.3"],
)
def load_and_split_data(
    test_size: float,
    random_state: int,
    n_samples: int,
    train_dataset: Output[Dataset],
    test_dataset: Output[Dataset],
):
    """Fetch the Covertype dataset, subsample, and split into train/test."""
    import pandas as pd
    from sklearn.datasets import fetch_covtype
    from sklearn.model_selection import train_test_split

    # Fetch full dataset (~580k rows) and subsample for speed
    data = fetch_covtype(as_frame=True)
    df = data.frame
    df = df.sample(n=min(n_samples, len(df)), random_state=random_state)

    # Split
    train_df, test_df = train_test_split(
        df, test_size=test_size, random_state=random_state, stratify=df['Cover_Type']
    )

    # Write to artifact paths — KFP handles GCS upload
    train_df.to_csv(train_dataset.path, index=False)
    test_df.to_csv(test_dataset.path, index=False)

    print(f"✅ Data loaded: {len(train_df)} train / {len(test_df)} test samples")

print("✅ load_and_split_data component defined.")

### 2.2 Training component

Now it's your turn. Write a component that trains a **`RandomForestClassifier`** from scikit-learn.

The component should:
- Accept `train_dataset` as `Input[Dataset]` and hyperparameters as simple types (`int`, `float`)
- Output the trained model as `Output[Model]`
- Store training metadata (hyperparameters, framework) on the model artifact

> 📖 **Docs:** [Artifact metadata](https://www.kubeflow.org/docs/components/pipelines/user-guides/data-handling/artifacts/) — use `artifact.metadata["key"] = value` to attach metadata to an artifact.

In [ ]:
# ✅ SOLUTION
@dsl.component(
    base_image="python:3.10",
    packages_to_install=["scikit-learn==1.5.2", "pandas==2.2.3"],
)
def train_model(
    train_dataset: Input[Dataset],
    n_estimators: int,
    max_depth: int,
    random_state: int,
    model_artifact: Output[Model],
):
    """Train a RandomForestClassifier on the training set."""
    import pickle
    import pandas as pd
    from sklearn.ensemble import RandomForestClassifier

    train_df = pd.read_csv(train_dataset.path)
    X_train = train_df.drop(columns=["Cover_Type"])
    y_train = train_df["Cover_Type"]

    clf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=random_state,
    )
    clf.fit(X_train, y_train)

    with open(model_artifact.path, "wb") as f:
        pickle.dump(clf, f)

    model_artifact.metadata["n_estimators"] = n_estimators
    model_artifact.metadata["max_depth"] = max_depth
    model_artifact.metadata["framework"] = "scikit-learn"

    print(f"✅ Model trained: n_estimators={n_estimators}, max_depth={max_depth}")

print("✅ train_model component defined.")

### 2.3 Evaluation component

Write a component that evaluates the trained model on the test set. It should:
- Load the test data and the pickled model
- Compute **accuracy** and **weighted F1 score**
- Log both metrics to the `Metrics` artifact

> 📖 **Docs:** [`Metrics` artifact](https://www.kubeflow.org/docs/components/pipelines/user-guides/data-handling/artifacts/#metrics) — use `metrics.log_metric(name, value)` to record evaluation metrics.

In [ ]:
# ✅ SOLUTION
@dsl.component(
    base_image="python:3.10",
    packages_to_install=["scikit-learn==1.5.2", "pandas==2.2.3"],
)
def evaluate_model(
    test_dataset: Input[Dataset],
    model_artifact: Input[Model],
    metrics: Output[Metrics],
):
    """Evaluate the trained model and log metrics."""
    import pickle
    import pandas as pd
    from sklearn.metrics import accuracy_score, f1_score

    test_df = pd.read_csv(test_dataset.path)
    X_test = test_df.drop(columns=["Cover_Type"])
    y_test = test_df["Cover_Type"]

    with open(model_artifact.path, "rb") as f:
        clf = pickle.load(f)

    y_pred = clf.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="weighted")

    metrics.log_metric("accuracy", round(accuracy, 4))
    metrics.log_metric("f1_weighted", round(f1, 4))

    print(f"✅ Evaluation — accuracy: {accuracy:.4f}, F1: {f1:.4f}")

print("✅ evaluate_model component defined.")

**✏️ Question 1 — Component isolation**

a) Why must all imports be placed **inside** the component function body, rather than at the top of the notebook?

b) A teammate suggests putting data loading, training, and evaluation in a **single** component. What are the drawbacks of this approach from an MLOps perspective?

---
*✅ Solution:*

a) Each component runs in its **own isolated container** — a fresh Python environment built from the `base_image`. The container has no knowledge of the notebook's environment. Imports inside the function ensure the component is self-contained and portable.

b) A monolithic component breaks the **single responsibility principle**. You lose: (1) **independent caching** — if data doesn't change, only training should re-run; (2) **parallel execution** — evaluation could run on a different node; (3) **reusability** — you can't reuse the evaluation component in another pipeline; (4) **debuggability** — if training fails, you'd have to re-run data loading too.

---

---
## 3 · Compose & Compile the Pipeline

Now that the components are defined, we wire them together into a **pipeline**. The `@dsl.pipeline` decorator marks a function as a pipeline definition. Inside, you instantiate components as tasks and pass outputs from one task as inputs to the next.

> 📖 **Docs:** [`@dsl.pipeline` decorator](https://www.kubeflow.org/docs/components/pipelines/user-guides/core-functions/compile-a-pipeline/)

### 3.1 Define the pipeline

The pipeline function receives **parameters** (hyperparameters, configuration) as arguments. These become the knobs you can turn when re-running the pipeline without changing the code.

In [ ]:
# ✅ SOLUTION
@dsl.pipeline(
    name="covertype-training-pipeline",
    description="Train and evaluate a RandomForest on the Covertype dataset.",
)
def covertype_pipeline(
    n_estimators: int = 100,
    max_depth: int = 10,
    test_size: float = 0.2,
    n_samples: int = 20000,
    random_state: int = 42,
):

    # Step 1: Load and split data
    data_task = load_and_split_data(
        test_size=test_size,
        random_state=random_state,
        n_samples=n_samples,
    )

    # Step 2: Train model
    train_task = train_model(
        train_dataset=data_task.outputs["train_dataset"],
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=random_state,
    )

    # Step 3: Evaluate model
    eval_task = evaluate_model(
        test_dataset=data_task.outputs["test_dataset"],
        model_artifact=train_task.outputs["model_artifact"],
    )

print("✅ Pipeline defined.")

### 3.2 Compile the pipeline

Compilation converts the Python pipeline definition into a **YAML** specification that Vertex AI Pipelines can execute. This YAML is portable — you can share it, version-control it, and schedule it.

In [ ]:
# ✅ SOLUTION
compiler.Compiler().compile(
    pipeline_func=covertype_pipeline,
    package_path="covertype_pipeline.yaml",
)

print("✅ Pipeline compiled to covertype_pipeline.yaml")

In [ ]:
# Inspect the first 40 lines of the compiled YAML
with open("covertype_pipeline.yaml", "r") as f:
    for i, line in enumerate(f):
        if i >= 40:
            print("...")
            break
        print(line, end='')

**✏️ Question 2 — Pipeline compilation**

a) Why does KFP compile the pipeline to YAML instead of executing the Python function directly?

b) What information can you identify in the YAML file? (Look at the first 40 lines.)

---
*✅ Solution:*

a) Compilation to YAML **decouples definition from execution**. The YAML is a portable, language-agnostic specification that can be: (1) stored in version control for reproducibility; (2) submitted to any compatible orchestrator (Vertex AI, standalone Kubeflow, etc.); (3) scheduled and triggered by external systems without needing the original Python code; (4) reviewed and audited as a declarative artifact.

b) The YAML contains: the pipeline name and description, the list of components with their container images and packages, the input/output specifications and types, the DAG structure (which task depends on which), and the default parameter values.

---

---
## 4 · Run on Vertex AI Pipelines

Now we submit the compiled pipeline to Vertex AI for execution. Each component will run in its own container on Google Cloud infrastructure.

> 📖 **Docs:** [`aiplatform.PipelineJob`](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform.PipelineJob)

### 4.1 Submit the pipeline

In [ ]:
# ✅ SOLUTION
job = aiplatform.PipelineJob(
    display_name=PIPELINE_DISPLAY_NAME,
    template_path="covertype_pipeline.yaml",
    pipeline_root=PIPELINE_ROOT,
    parameter_values={},
)
job.run()

print(f"✅ Pipeline completed: {job.display_name}")
print(f"   Job resource name: {job.resource_name}")

### 4.2 Monitor execution

You can monitor the pipeline execution in the **Vertex AI console**.

> 💡 **Tip:** Go to the Vertex AI console → **Pipelines** section to see the DAG visualization, logs for each component, and artifact details.

**✏️ Question 3 — Pipeline execution**

a) In the Vertex AI console, examine the pipeline DAG. How does the execution order of the components match the data dependencies you defined?

b) What happens if the `train_model` component fails? Does the `evaluate_model` component still run? Why or why not?

---
*✅ Solution:*

a) `load_and_split_data` runs first because it has no upstream dependencies. `train_model` starts only after `load_and_split_data` completes (it needs `train_dataset`). `evaluate_model` starts after **both** `load_and_split_data` (for `test_dataset`) and `train_model` (for `model_artifact`) complete.

b) No. If `train_model` fails, `evaluate_model` is **skipped** because it depends on `model_artifact` which was never produced. KFP follows the DAG — a task only runs when all its upstream dependencies have completed successfully. This is a key benefit of pipeline orchestration: failures are contained and downstream tasks are not wasted.

---

---
## 5 · Parameterize & Re-run

A major advantage of pipelines over notebooks is **parameterization**: you can re-run the same pipeline with different configurations without touching any code. This is essential for hyperparameter tuning and reproducible experiments.

### 5.1 Run with different hyperparameters

In [ ]:
# ✅ SOLUTION
job_2 = aiplatform.PipelineJob(
    display_name=f"{PIPELINE_DISPLAY_NAME}-v2",
    template_path="covertype_pipeline.yaml",
    pipeline_root=PIPELINE_ROOT,
    parameter_values={
        "n_estimators": 200,
        "max_depth": 20,
    },
)
job_2.run()

print(f"✅ Second pipeline completed: {job_2.display_name}")

### 5.2 Compare pipeline runs

You can list and compare pipeline runs programmatically.

In [ ]:
# List recent pipeline jobs
jobs = aiplatform.PipelineJob.list(
    filter=f'display_name="{PIPELINE_DISPLAY_NAME}" OR display_name="{PIPELINE_DISPLAY_NAME}-v2"'
)

for j in jobs:
    params = j.pipeline_spec.get('runtimeConfig', {}).get('parameterValues', {})
    print(f"Run: {j.display_name:<40} State: {j.state.name:<20}")
    print(f"     Parameters: {params}")
    print()

> 💡 **Tip:** In the Vertex AI console → Pipelines, click on each run to compare the evaluation metrics logged by the `evaluate_model` component. You can see them in the **metrics** artifcat's details of each run.

**✏️ Question 4 — Parameterization & reproducibility**

a) Why are one of the pipelines parameters printed as an empty dict?

b) Why is it better to change hyperparameters through `parameter_values` rather than editing the component code and recompiling?

c) How would you integrate pipeline parameterization with the Vertex AI Experiments you used in Lab 02a?

---
*✅ Solution:*

a)`j.gca_resource.runtime_config.parameter_values` is a proto map that contains the actual values passed at submission time. The first run (with defaults) shows an empty dict since you passed {} — only the second run with explicit overrides shows the parameters. That's actually correct behavior: default values live in the YAML spec, not in the runtime config.

b) Parameterization via `parameter_values` means the **same compiled YAML** is reused across runs. This guarantees the pipeline logic is identical and only the configuration changes. Editing code + recompiling risks introducing unintended changes and breaks the link between runs. It also enables automation: a scheduler or CI/CD system can trigger runs with different parameters without touching the pipeline code.

c) You could add Vertex AI experiment tracking **inside** a component: call `aiplatform.init(experiment=...)`, `aiplatform.start_run(...)`, and log the pipeline's parameters and evaluation metrics. This would let you compare pipeline runs alongside notebook-based experiments in the same Vertex AI Experiments dashboard, giving a unified view across all approaches.

---

---
## 6 · Cleanup

Delete the pipeline runs to avoid unnecessary storage costs.

In [ ]:
# Delete both pipeline jobs
for j in [job, job_2]:
    try:
        j.delete()
        print(f"✅ Deleted: {j.display_name}")
    except Exception as e:
        print(f"⚠️ Could not delete {j.display_name}: {e}")

---
## Summary

In this lab, you learned to:

| Step | What you did | Tool / Feature used |
|------|-------------|---------------------|
| Components | Built lightweight Python components with typed I/O | `@dsl.component`, `Input`, `Output`, `Dataset`, `Model`, `Metrics` |
| Pipeline | Composed components into a DAG | `@dsl.pipeline` |
| Compilation | Compiled the pipeline to a portable YAML file | `compiler.Compiler().compile()` |
| Execution | Submitted the pipeline to Vertex AI Pipelines | `aiplatform.PipelineJob` |
| Parameterization | Re-ran the pipeline with different hyperparameters | `parameter_values` |

**Next lab:** In Lab 04b, you will build **production pipeline patterns**: data validation gates, conditional deployment logic, and model registration — turning a simple pipeline into a production-grade ML workflow.